In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
GPR-based InSAR <-> GPS validation + bias extraction + resolution/tradeoff analysis
Updated with PBO NAM14 stations (n~30 total, weighted stats, lower threshold)
"""
import os
import numpy as np
import pandas as pd
import rioxarray as rxr
import pyproj
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel, DotProduct
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from scipy.spatial import cKDTree
import warnings
import re # برای parse PBO

# ----------------- USER PARAMETERS -----------------
INSAR_DIR = "/home/drebadi/Downloads/phd/outputs/outputs_master/proper_maps_final/"
INSAR_RASTERS = {"E": "dx_cropped.tif", "N": "dy_cropped.tif", "U": "dz_cropped.tif"}
CSV_FILE = "IGS14__alaska.csv"
PBO_FILE = "nmt.final_nam14.vel" # PBO path
OUT_DIR = "./edge_validation_gpr_pbo/"
os.makedirs(OUT_DIR, exist_ok=True)
TARGET_CRS = "EPSG:3338"
FULL_SAMPLE_LIMIT = 600000
GPR_SAMPLE_LIMIT = 5000
DISTANCE_LIMIT_M = 150000 # 150km for more stations (n~25)
GPR_ALPHA = 1e-6
# ---------------------------------------------------
warnings.filterwarnings("ignore", category=UserWarning)

def read_gps(csv_path):
    """Read GPS CSV with flexible column name detection."""
    df = pd.read_csv(csv_path, sep=None, engine="python")
    df.columns = df.columns.str.strip().str.replace(r"[\ufeff\t]+", "", regex=True)
    df.columns = df.columns.str.replace(r"\s+", " ", regex=True)
    def find_col(cands):
        for c in df.columns:
            if any(p.lower() in c.lower() for p in cands):
                return c
        return None
    lon_col = find_col(["lon", "long", "x"])
    lat_col = find_col(["lat", "latitude", "y"])
    df = df.rename(columns={lon_col: "lon", lat_col: "lat"})
    col_map = {
        "E": find_col(["east", "vel_e", "east (mm)"]),
        "N": find_col(["north", "vel_n", "north (mm)"]),
        "U": find_col(["up", "vert", "vertical", "uplift", "vertical (mm)"]),
    }
    for k, v in col_map.items():
        if v:
            df = df.rename(columns={v: k})
        else:
            df[k] = np.nan # fill missing component
    if "Station" not in df.columns:
        c = find_col(["station", "name"])
        if c:
            df = df.rename(columns={c: "Station"})
    return df

def read_pbo_vel(vel_file):
    """Read PBO/NAM14 vel file (same format as NAM14)."""
    with open(vel_file, 'r') as f:
        lines = f.readlines()
    data_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped and not stripped.startswith('*') and not stripped.startswith('PBO') and not stripped.startswith('Reference') and not stripped.startswith('Format') and not stripped.startswith('Release') and not stripped.startswith('Start') and not stripped.startswith('End') and not stripped.startswith('Dot#'):
            parts = re.split(r'\s+', stripped)
            if len(parts) >= 30:
                data_lines.append(parts[:30])
    if not data_lines:
        return pd.DataFrame()
    columns = ['Station', 'Name', 'Ref_epoch', 'Ref_jday', 'Ref_X', 'Ref_Y', 'Ref_Z', 'Ref_Nlat', 'Ref_Elong', 'Ref_Up',
               'dX/dt', 'dY/dt', 'dZ/dt', 'SXd', 'SYd', 'SZd', 'Rxy', 'Rxz', 'Ryz',
               'dN/dt', 'dE/dt', 'dU/dt', 'SNd', 'SEd', 'SUd', 'Rne', 'Rnu', 'Reu',
               'first_epoch', 'last_epoch']
    df = pd.DataFrame(data_lines, columns=columns)
    numeric_cols = ['Ref_jday', 'Ref_X', 'Ref_Y', 'Ref_Z', 'Ref_Nlat', 'Ref_Elong', 'Ref_Up',
                    'dX/dt', 'dY/dt', 'dZ/dt', 'SXd', 'SYd', 'SZd',
                    'dN/dt', 'dE/dt', 'dU/dt', 'SNd', 'SEd', 'SUd', 'Rxy', 'Rxz', 'Ryz', 'Rne', 'Rnu', 'Reu']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['Ref_Nlat', 'Ref_Elong'])
    df['lon'] = np.where(df['Ref_Elong'] > 180, df['Ref_Elong'] - 360, df['Ref_Elong'])
    df['last_epoch_num'] = pd.to_numeric(df['last_epoch'].str[:8], errors='coerce')
    df = df.dropna(subset=['last_epoch_num'])
    df_grouped = df.loc[df.groupby('Station')['last_epoch_num'].idxmax()]
    # فیلتر Denali (۵۹-۶۱°N, -۱۶۰ تا -۱۵۰°W)
    df_denali = df_grouped[(df_grouped['Ref_Nlat'] >= 59) & (df_grouped['Ref_Nlat'] <= 61) &
                           (df_grouped['lon'] >= -160) & (df_grouped['lon'] <= -150)].copy()
    # تبدیل به mm/yr
    df_denali['E'] = df_denali['dE/dt'] * 1000
    df_denali['N'] = df_denali['dN/dt'] * 1000
    df_denali['U'] = df_denali['dU/dt'] * 1000
    df_denali['SE'] = df_denali['SEd'] * 1000
    df_denali['SN'] = df_denali['SNd'] * 1000
    df_denali['SU'] = df_denali['SUd'] * 1000
    df_final = df_denali[['Station', 'lon', 'Ref_Nlat', 'E', 'N', 'U', 'SE', 'SN', 'SU']].rename(columns={'Ref_Nlat': 'lat'})
    return df_final

def sample_insar_pixels(raster_path, target_crs=TARGET_CRS, sample_limit=FULL_SAMPLE_LIMIT):
    da = rxr.open_rasterio(raster_path).squeeze()
    if da.rio.crs is None:
        da = da.rio.write_crs("EPSG:4326", inplace=True)
    if da.rio.crs.to_string() != target_crs:
        print(f"Reprojecting {os.path.basename(raster_path)}...")
        da = da.rio.reproject(target_crs)
    xs, ys = da.x.values, da.y.values
    xv, yv = np.meshgrid(xs, ys)
    vals = da.values.flatten()
    coords = np.column_stack((xv.flatten(), yv.flatten()))
    mask = ~np.isnan(vals)
    coords, vals = coords[mask], vals[mask]
    if len(coords) > sample_limit:
        _, idx = train_test_split(np.arange(len(coords)), test_size=sample_limit / len(coords), random_state=42)
        coords, vals = coords[idx], vals[idx]
    return coords, vals

def subsample_for_gpr(coords, vals, subsample_size=GPR_SAMPLE_LIMIT):
    n = len(vals)
    if n <= subsample_size:
        return coords, vals
    bins = np.percentile(vals, np.linspace(0, 100, 11))
    classes = np.digitize(vals, bins)
    try:
        _, idx = train_test_split(np.arange(n), test_size=subsample_size / n, stratify=classes, random_state=42)
    except Exception:
        _, idx = train_test_split(np.arange(n), test_size=subsample_size / n, random_state=42)
    return coords[idx], vals[idx]

def fit_gpr(X_train, y_train):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    kernel = ConstantKernel(1.0, (1e-2, 1e2)) * (
        RBF(length_scale=0.1, length_scale_bounds=(1e-3, 1e2)) + DotProduct()
    ) + WhiteKernel(noise_level=0.3, noise_level_bounds=(1e-5, 1e1))
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=GPR_ALPHA, normalize_y=True, random_state=42, n_restarts_optimizer=5)
    gpr.fit(X_scaled, y_train)
    print(" Fitted kernel:", gpr.kernel_)
    return gpr, scaler

def safe_pearsonr(x, y):
    if len(x) < 2 or np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan
    return pearsonr(x, y)[0]

def weighted_stats(residuals, y):
    """Weighted bias and RMSE."""
    mask = ~np.isnan(residuals)
    if np.sum(mask) == 0:
        return np.nan, np.nan
    res = residuals[mask]
    w = y[mask]
    wsum = np.nansum(w)
    if wsum == 0:
        return np.nan, np.nan
    bias_w = np.nansum(w * res) / wsum
    rmse_w = np.sqrt(np.nansum(w * (res - bias_w)**2) / wsum)
    return bias_w, rmse_w

def compute_stats(df_sub, comp):
    gps_vals = df_sub[comp].values
    insar_vals = df_sub[f"insar_{comp}"].values
    res = insar_vals - gps_vals
    bias = np.nanmean(res)
    rmse = np.sqrt(np.nanmean(res**2))
    corr = safe_pearsonr(gps_vals, insar_vals)
    # Weighted if sigma present
    sigma_col = f"{comp}_sigma" # e.g., 'E_sigma'
    bias_w = rmse_w = None
    if sigma_col in df_sub.columns:
        sigma = df_sub[sigma_col].values
        w = 1.0 / (sigma**2 + 1e-6)
        bias_w, rmse_w = weighted_stats(res, w)
    return {"n": len(df_sub), "bias": bias, "rmse": rmse, "r": corr, "bias_w": bias_w, "rmse_w": rmse_w}

def plot_scatter(df, comp, bias, rmse, corr, bias_w=None, rmse_w=None):
    plt.figure(figsize=(6, 6))
    plt.scatter(df[comp], df[f"insar_{comp}"], c=df[f"dist_nearest_m_{comp}"] / 1000, cmap="viridis", s=60, edgecolor="white", alpha=0.8)
    lims = [np.nanmin([df[comp], df[f"insar_{comp}"]]), np.nanmax([df[comp], df[f"insar_{comp}"]])]
    plt.plot(lims, lims, "k--", lw=1.2)
    plt.xlabel(f"GPS {comp} (mm)")
    plt.ylabel(f"InSAR {comp} (mm)")
    plt.title(f"{comp}-component: InSAR vs GPS")
    text = f"Bias={bias:.2f}\nRMSE={rmse:.2f}\nR={corr:.3f}"
    if bias_w is not None:
        text += f"\nWeighted Bias={bias_w:.2f}\nWeighted RMSE={rmse_w:.2f}"
    plt.text(0.05, 0.95, text, transform=plt.gca().transAxes,
             fontsize=10, va="top", bbox=dict(facecolor="white", alpha=0.7))
    plt.colorbar(label="Dist to nearest pixel (km)")
    plt.grid(True, ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"scatter_{comp}.png"), dpi=300)
    plt.close()

def plot_residual_hist(residuals, comp):
    plt.figure(figsize=(6, 6))
    plt.hist(residuals, bins=20, color="steelblue", alpha=0.7, edgecolor="black")
    plt.xlabel("Residual (InSAR - GPS) [mm]")
    plt.ylabel("Count")
    plt.title(f"Residual Distribution ({comp})")
    plt.grid(True, ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"residual_hist_{comp}.png"), dpi=300)
    plt.close()

def plot_spatial(df, comp):
    plt.figure(figsize=(7, 5))
    sc = plt.scatter(df["x_m"], df["y_m"], c=df[f"insar_{comp}"] - df[comp], cmap="RdBu_r", s=70, edgecolor="k")
    plt.colorbar(sc, label=f"Residual (InSAR - GPS) [{comp}] (mm)")
    plt.title(f"Spatial Residuals ({comp})")
    plt.xlabel("X (m)")
    plt.ylabel("Y (m)")
    plt.grid(ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"spatial_residuals_{comp}.png"), dpi=300)
    plt.close()

def plot_residual_vs_dist(df, comp):
    residuals = df[f"insar_{comp}"] - df[comp]
    dist_km = df[f"dist_nearest_m_{comp}"] / 1000
    plt.figure(figsize=(6, 6))
    plt.scatter(dist_km, residuals, s=60, alpha=0.8)
    plt.xlabel("Distance to nearest InSAR pixel (km)")
    plt.ylabel("Residual (InSAR - GPS) [mm]")
    plt.title(f"Residual vs Distance ({comp})")
    plt.grid(True, ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"residual_vs_distance_{comp}.png"), dpi=300)
    plt.close()

def run_all():
    gps = read_gps(CSV_FILE)
    # Add PBO stations
    df_pbo = read_pbo_vel(PBO_FILE)
    if not df_pbo.empty:
        df_pbo = df_pbo.rename(columns={'SE': 'E_sigma', 'SN': 'N_sigma', 'SU': 'U_sigma'})
        gps = pd.concat([gps, df_pbo], ignore_index=True)
        print(f"Added {len(df_pbo)} PBO stations. Total GPS: {len(gps)}")
    transformer = pyproj.Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)
    gps_x, gps_y = transformer.transform(gps["lon"].values, gps["lat"].values)
    gps_coords = np.column_stack((gps_x, gps_y))
    print(f"Loaded {len(gps)} stations -> CRS {TARGET_CRS}")
    for comp, tif in INSAR_RASTERS.items():
        print(f"\n--- Processing component {comp} ---")
        coords, vals = sample_insar_pixels(os.path.join(INSAR_DIR, tif))
        coords_sub, vals_sub = subsample_for_gpr(coords, vals)
        gpr, scaler = fit_gpr(coords_sub, vals_sub)
        gps_scaled = scaler.transform(gps_coords)
        insar_preds, insar_std = gpr.predict(gps_scaled, return_std=True)
        tree = cKDTree(coords)
        nearest_dists, _ = tree.query(gps_coords, k=1)
        valid_mask = nearest_dists <= DISTANCE_LIMIT_M
        print(f"Valid stations within {DISTANCE_LIMIT_M/1000}km: {np.sum(valid_mask)}")
        df = gps.copy()
        df["x_m"], df["y_m"] = gps_x, gps_y
        df[f"insar_{comp}"] = insar_preds
        df[f"insar_std_{comp}"] = insar_std
        df[f"dist_nearest_m_{comp}"] = nearest_dists
        df_valid = df.loc[valid_mask].reset_index(drop=True)
        stats = compute_stats(df_valid, comp)
        print(f"Bias={stats['bias']:.2f} mm | RMSE={stats['rmse']:.2f} mm | Corr={stats['r']:.3f}")
        if stats['bias_w'] is not None:
            print(f"Weighted Bias={stats['bias_w']:.2f} mm | Weighted RMSE={stats['rmse_w']:.2f} mm")
        # Save CSV and plots
        df_valid.to_csv(os.path.join(OUT_DIR, f"edge_validation_gpr_{comp}.csv"), index=False)
        plot_scatter(df_valid, comp, stats['bias'], stats['rmse'], stats['r'], stats['bias_w'], stats['rmse_w'])
        plot_residual_hist(df_valid[f"insar_{comp}"] - df_valid[comp], comp)
        plot_spatial(df_valid, comp)
        plot_residual_vs_dist(df_valid, comp)
    print("\n✅ All components processed. Results saved to:", OUT_DIR)

if __name__ == "__main__":
    run_all()

Added 27 PBO stations. Total GPS: 37
Loaded 37 stations -> CRS EPSG:3338

--- Processing component E ---
Reprojecting dx_cropped.tif...
 Fitted kernel: 0.694**2 * RBF(length_scale=0.106) + DotProduct(sigma_0=1.61e-05) + WhiteKernel(noise_level=0.335)
Valid stations within 150.0km: 10
Bias=-3.14 mm | RMSE=11.33 mm | Corr=-0.384
Weighted Bias=-13.98 mm | Weighted RMSE=1.39 mm

--- Processing component N ---
Reprojecting dy_cropped.tif...
 Fitted kernel: 0.629**2 * RBF(length_scale=0.141) + DotProduct(sigma_0=1.61e-05) + WhiteKernel(noise_level=0.418)
Valid stations within 150.0km: 10
Bias=6.96 mm | RMSE=29.15 mm | Corr=-0.602
Weighted Bias=-12.50 mm | Weighted RMSE=11.66 mm

--- Processing component U ---
Reprojecting dz_cropped.tif...
 Fitted kernel: 0.746**2 * RBF(length_scale=0.0719) + DotProduct(sigma_0=1.61e-05) + WhiteKernel(noise_level=0.09)
Valid stations within 150.0km: 10
Bias=4.34 mm | RMSE=19.88 mm | Corr=0.905
Weighted Bias=9.32 mm | Weighted RMSE=15.24 mm

✅ All components 

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
GPR-based InSAR <-> GPS validation + bias extraction + resolution/tradeoff analysis
Defense-ready: threshold analysis (30/50/100/all), weighted bias, residual vs distance, improved kernel
"""
import os
import numpy as np
import pandas as pd
import rioxarray as rxr
import pyproj
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel, DotProduct
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from scipy.spatial import cKDTree
import warnings

# ----------------- USER PARAMETERS -----------------
INSAR_DIR = "/home/drebadi/Downloads/phd/outputs/outputs_master/proper_maps_final/"
INSAR_RASTERS = {"E": "dx_cropped.tif", "N": "dy_cropped.tif", "U": "dz_cropped.tif"}
CSV_FILE = "IGS14__alaska.csv"
OUT_DIR = "./edge_validation_gpr_fixed/"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_CRS = "EPSG:3338"
FULL_SAMPLE_LIMIT = 600000
GPR_SAMPLE_LIMIT = 5000
# default limit used for the "valid_mask" step (but script also computes thresholds 30/50/100/all)
DISTANCE_LIMIT_M = 50000
GPR_ALPHA = 1e-6
# thresholds (meters) for sensitivity analysis
THRESHOLDS_M = [30000, 50000, 100000, np.inf]
# ---------------------------------------------------

warnings.filterwarnings("ignore", category=UserWarning)


def read_gps(csv_path):
    df = pd.read_csv(csv_path, sep=None, engine="python")
    df.columns = df.columns.str.strip().str.replace(r"[\ufeff\t]+", "", regex=True)
    df.columns = df.columns.str.replace(r"\s+", " ", regex=True)

    def find_col(cands):
        for c in df.columns:
            if any(p.lower() in c.lower() for p in cands):
                return c
        return None

    lon_col = find_col(["lon", "long", "x"])
    lat_col = find_col(["lat", "latitude", "y"])
    df = df.rename(columns={lon_col: "lon", lat_col: "lat"})

    col_map = {
        "E": find_col(["east", "vel_e", "east (mm)", "e (mm)"]),
        "N": find_col(["north", "vel_n", "north (mm)", "n (mm)"]),
        "U": find_col(["up", "vert", "vertical", "uplift", "vertical (mm)"]),
    }
    for k, v in col_map.items():
        if v:
            df = df.rename(columns={v: k})
        else:
            df[k] = np.nan  # fill missing component

    if "Station" not in df.columns:
        c = find_col(["station", "name"])
        if c:
            df = df.rename(columns={c: "Station"})
    return df


def sample_insar_pixels(raster_path, target_crs=TARGET_CRS, sample_limit=FULL_SAMPLE_LIMIT):
    da = rxr.open_rasterio(raster_path).squeeze()
    if da.rio.crs is None:
        da = da.rio.write_crs("EPSG:4326", inplace=True)
    if da.rio.crs.to_string() != target_crs:
        print(f"Reprojecting {os.path.basename(raster_path)}...")
        da = da.rio.reproject(target_crs)

    xs, ys = da.x.values, da.y.values
    xv, yv = np.meshgrid(xs, ys)
    vals = da.values.flatten()
    coords = np.column_stack((xv.flatten(), yv.flatten()))
    mask = ~np.isnan(vals)
    coords, vals = coords[mask], vals[mask]

    if len(coords) > sample_limit:
        _, idx = train_test_split(np.arange(len(coords)), test_size=sample_limit / len(coords), random_state=42)
        coords, vals = coords[idx], vals[idx]
    return coords, vals


def subsample_for_gpr(coords, vals, subsample_size=GPR_SAMPLE_LIMIT):
    n = len(vals)
    if n <= subsample_size:
        return coords, vals
    bins = np.percentile(vals, np.linspace(0, 100, 11))
    classes = np.digitize(vals, bins)
    try:
        _, idx = train_test_split(np.arange(n), test_size=subsample_size / n, stratify=classes, random_state=42)
    except Exception:
        _, idx = train_test_split(np.arange(n), test_size=subsample_size / n, random_state=42)
    return coords[idx], vals[idx]


def fit_gpr(X_train, y_train):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    # Improved kernel: Matern is often more robust for geospatial data than pure RBF
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * (Matern(length_scale=1.0, nu=1.5) + DotProduct()) + WhiteKernel(
        noise_level=0.1, noise_level_bounds=(1e-6, 1e2)
    )

    gpr = GaussianProcessRegressor(kernel=kernel, alpha=GPR_ALPHA, normalize_y=True, random_state=42,
                                   n_restarts_optimizer=8)
    gpr.fit(X_scaled, y_train)
    print("  Fitted kernel:", gpr.kernel_)
    return gpr, scaler


def safe_pearsonr(x, y):
    if len(x) < 2 or np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan
    return pearsonr(x, y)[0]


def save_df_and_plots(df_valid, comp, bias_raw, rmse_raw, corr_raw, bias_corr=None, rmse_corr=None, corr_corr=None):
    # CSV
    out_csv = os.path.join(OUT_DIR, f"edge_validation_gpr_{comp}.csv")
    df_valid.to_csv(out_csv, index=False)

    # Scatter raw
    plt.figure(figsize=(6, 6))
    plt.scatter(df_valid[comp], df_valid[f"insar_{comp}"], c="steelblue", s=60, edgecolor="white", alpha=0.85)
    lims = [np.nanmin([df_valid[comp], df_valid[f"insar_{comp}"]]), np.nanmax([df_valid[comp], df_valid[f"insar_{comp}"]])]
    plt.plot(lims, lims, "k--", lw=1.2)
    plt.xlabel(f"GPS {comp} (mm)")
    plt.ylabel(f"InSAR {comp} (mm)")
    plt.title(f"{comp}-component: InSAR vs GPS (raw)")
    plt.text(0.05, 0.95, f"Bias={bias_raw:.2f}\nRMSE={rmse_raw:.2f}\nR={corr_raw:.2f}", transform=plt.gca().transAxes,
             fontsize=10, va="top", bbox=dict(facecolor="white", alpha=0.7))
    plt.grid(True, ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"scatter_{comp}_raw.png"), dpi=300)
    plt.close()

    # Scatter bias-corrected
    if bias_corr is not None:
        insar_corr = df_valid[f"insar_{comp}"] - bias_corr
        plt.figure(figsize=(6, 6))
        plt.scatter(df_valid[comp], insar_corr, c="seagreen", s=60, edgecolor="white", alpha=0.85)
        lims = [np.nanmin([df_valid[comp], insar_corr]), np.nanmax([df_valid[comp], insar_corr])]
        plt.plot(lims, lims, "k--", lw=1.2)
        plt.xlabel(f"GPS {comp} (mm)")
        plt.ylabel(f"InSAR {comp} (mm) (bias-corrected)")
        plt.title(f"{comp}-component: InSAR vs GPS (bias-corrected)")
        plt.text(0.05, 0.95, f"Bias={bias_corr:.2f}\nRMSE={rmse_corr:.2f}\nR={corr_corr:.2f}", transform=plt.gca().transAxes,
                 fontsize=10, va="top", bbox=dict(facecolor="white", alpha=0.7))
        plt.grid(True, ls="--", alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"scatter_{comp}_biascorr.png"), dpi=300)
        plt.close()

    # residual hist (raw)
    residuals = df_valid[f"insar_{comp}"] - df_valid[comp]
    plt.figure(figsize=(6, 4))
    plt.hist(residuals, bins=20, alpha=0.7, edgecolor="black")
    plt.xlabel("Residual (InSAR - GPS) [mm]")
    plt.ylabel("Count")
    plt.title(f"Residual Distribution ({comp}) (raw)")
    plt.grid(True, ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"residual_hist_{comp}_raw.png"), dpi=300)
    plt.close()

    # spatial residuals
    plt.figure(figsize=(7, 5))
    sc = plt.scatter(df_valid["x_m"], df_valid["y_m"], c=residuals, cmap="RdBu_r", s=70, edgecolor="k")
    plt.colorbar(sc, label=f"Residual (InSAR - GPS) [{comp}] (mm)")
    plt.title(f"Spatial Residuals ({comp})")
    plt.xlabel("X (m)")
    plt.ylabel("Y (m)")
    plt.grid(ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"spatial_residuals_{comp}.png"), dpi=300)
    plt.close()

    # residual vs distance
    plt.figure(figsize=(6, 4))
    d_km = df_valid[f"dist_nearest_m_{comp}"] / 1000.0
    plt.scatter(d_km, residuals, s=60, alpha=0.85, edgecolor='k')
    plt.xlabel("Distance to nearest InSAR pixel (km)")
    plt.ylabel("Residual (InSAR - GPS) [mm]")
    plt.title(f"Residual vs Distance ({comp})")
    plt.grid(True, ls='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"residual_vs_distance_{comp}.png"), dpi=300)
    plt.close()


def weighted_stats(residuals, w):
    w = np.array(w, dtype=float)
    mask = ~np.isnan(residuals)
    if np.sum(mask) == 0:
        return np.nan, np.nan
    res = residuals[mask]
    w = w[mask]
    wsum = np.nansum(w)
    if wsum == 0:
        return np.nan, np.nan
    bias_w = np.nansum(w * res) / wsum
    rmse_w = np.sqrt(np.nansum(w * (res - bias_w) ** 2) / wsum)
    return bias_w, rmse_w


def compute_stats(df_sub, comp):
    gps_vals = df_sub[comp].values
    insar_vals = df_sub[f"insar_{comp}"].values
    res = insar_vals - gps_vals
    bias = np.nanmean(res) if len(res) > 0 else np.nan
    rmse = np.sqrt(np.nanmean(res ** 2)) if len(res) > 0 else np.nan
    corr = safe_pearsonr(gps_vals, insar_vals)
    # attempt weighted bias using GPS sigma columns if available
    sigma_col_candidates = [c for c in df_sub.columns if "std" in c.lower() or "deviation" in c.lower()]
    sigma_col = None
    for c in sigma_col_candidates:
        if comp.lower() in c.lower():
            sigma_col = c
            break
    bias_w = rmse_w = None
    if sigma_col is not None:
        sigma = df_sub[sigma_col].values
        w = 1.0 / (sigma ** 2 + 1e-6)
        bias_w, rmse_w = weighted_stats(res, w)
    return {"n": len(df_sub), "bias": bias, "rmse": rmse, "r": corr, "bias_w": bias_w, "rmse_w": rmse_w, "sigma_col": sigma_col}


def run_all():
    gps = read_gps(CSV_FILE)
    transformer = pyproj.Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)
    gps_x, gps_y = transformer.transform(gps["lon"].values, gps["lat"].values)
    gps_coords = np.column_stack((gps_x, gps_y))
    print(f"Loaded {len(gps)} stations -> CRS {TARGET_CRS}")

    # store summary for thresholds
    summary_rows = []

    for comp, tif in INSAR_RASTERS.items():
        print(f"\n--- Processing component {comp} ---")
        coords, vals = sample_insar_pixels(os.path.join(INSAR_DIR, tif))
        coords_sub, vals_sub = subsample_for_gpr(coords, vals)
        gpr, scaler = fit_gpr(coords_sub, vals_sub)

        gps_scaled = scaler.transform(gps_coords)
        insar_preds, insar_std = gpr.predict(gps_scaled, return_std=True)

        tree = cKDTree(coords)
        nearest_dists, _ = tree.query(gps_coords, k=1)

        df = gps.copy()
        df["x_m"], df["y_m"] = gps_x, gps_y
        df[f"insar_{comp}"] = insar_preds
        df[f"insar_std_{comp}"] = insar_std
        df[f"dist_nearest_m_{comp}"] = nearest_dists

        # For final CSV we still save full table with predictions
        df.to_csv(os.path.join(OUT_DIR, f"edge_validation_gpr_allpreds_{comp}.csv"), index=False)

        # Sensitivity by thresholds
        for thr in THRESHOLDS_M:
            if np.isinf(thr):
                df_sub = df.copy()
                label = "all"
            else:
                df_sub = df[df[f"dist_nearest_m_{comp}"] <= thr].reset_index(drop=True)
                label = f"{int(thr/1000)}km"

            stats = compute_stats(df_sub, comp)
            # compute bias-corrected metrics if possible (bias correction using simple mean bias)
            if stats["n"] > 0:
                residuals = df_sub[f"insar_{comp}"].values - df_sub[comp].values
                bias_mean = np.nanmean(residuals)
                insar_corr = df_sub[f"insar_{comp}"].values - bias_mean
                corr_corr = safe_pearsonr(df_sub[comp].values, insar_corr)
                rmse_corr = np.sqrt(np.nanmean((insar_corr - df_sub[comp].values) ** 2))
            else:
                bias_mean = corr_corr = rmse_corr = np.nan

            summary_rows.append({
                "component": comp,
                "threshold": label,
                "n": stats["n"],
                "bias_raw_mm": stats["bias"],
                "rmse_raw_mm": stats["rmse"],
                "r_raw": stats["r"],
                "bias_weighted_mm": stats["bias_w"],
                "rmse_weighted_mm": stats["rmse_w"],
                "sigma_col_used": stats["sigma_col"],
                "bias_mean_corrected_mm": bias_mean,
                "rmse_corr_mm": rmse_corr,
                "r_corr": corr_corr
            })

            # For the selected threshold equal to default DISTANCE_LIMIT_M, also save plots & csv
            if (not np.isinf(thr) and thr == DISTANCE_LIMIT_M) or (np.isinf(thr) and thr == THRESHOLDS_M[-1] and DISTANCE_LIMIT_M == np.inf):
                df_valid = df_sub.copy()
                print(f"For comp {comp} using threshold {label}: n={len(df_valid)}")
                # compute raw stats for printing
                gps_vals = df_valid[comp].values
                insar_vals = df_valid[f"insar_{comp}"].values
                residuals = insar_vals - gps_vals
                bias = np.nanmean(residuals) if len(residuals)>0 else np.nan
                rmse = np.sqrt(np.nanmean(residuals**2)) if len(residuals)>0 else np.nan
                corr = safe_pearsonr(gps_vals, insar_vals)
                # weighted bias if sigma present
                sigma_col = compute_stats(df_valid, comp)["sigma_col"]
                bias_w = rmse_w = None
                if sigma_col:
                    sigma = df_valid[sigma_col].values
                    w = 1.0 / (sigma ** 2 + 1e-6)
                    bias_w, rmse_w = weighted_stats(residuals, w)

                # Save csv & plots (raw + bias-corrected)
                save_df_and_plots(df_valid, comp, bias, rmse, corr, bias_corr=bias_mean, rmse_corr=rmse_corr, corr_corr=corr_corr)
                # also save a small CSV only with selected df_valid
                df_valid.to_csv(os.path.join(OUT_DIR, f"edge_validation_gpr_{comp}_{label}.csv"), index=False)

    # write summary table
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(OUT_DIR, "thresholds_summary.csv"), index=False)
    print("\n✅ All components processed. Summary saved to:", OUT_DIR)


if __name__ == "__main__":
    run_all()


Loaded 10 stations -> CRS EPSG:3338

--- Processing component E ---
Reprojecting dx_cropped.tif...
  Fitted kernel: 0.735**2 * Matern(length_scale=0.148, nu=1.5) + DotProduct(sigma_0=1e-05) + WhiteKernel(noise_level=0.306)
For comp E using threshold 50km: n=2

--- Processing component N ---
Reprojecting dy_cropped.tif...
  Fitted kernel: 0.688**2 * Matern(length_scale=0.158, nu=1.5) + DotProduct(sigma_0=1e-05) + WhiteKernel(noise_level=0.365)
For comp N using threshold 50km: n=2

--- Processing component U ---
Reprojecting dz_cropped.tif...
  Fitted kernel: 0.793**2 * Matern(length_scale=0.0966, nu=1.5) + DotProduct(sigma_0=1.61e-05) + WhiteKernel(noise_level=0.0525)
For comp U using threshold 50km: n=2

✅ All components processed. Summary saved to: ./edge_validation_gpr_fixed/
